# 35. Sequential sensitivity analysis: robustness of the development-period H1 result to alert-timing bias under a cooldown

In notebook 32 I found that the budget-preserving regime-matched null rejected 44 of 600 surrogate series with no predictive power (7.3%; nominal 5%), and in notebook 34 I showed with the exact conditional null Q that the resampler approximation does not cause this excess (these are observed rejection rates on surrogate series that are not independent, not a strict proof that the test exceeds the nominal level).
Here I relax the conditional placement assumption that alert times are exchangeable with respect to events within each cell (month × volatility decile) and ask how strong such timing bias would have to be to explain the M0 result in the development period (spot BTCUSDT one-minute data, 2023-01 to 2026-04): 22 pre-onset catches, exact p = 7.4e-8 in notebook 34.
I do not use Rosenbaum's model directly: under a cooldown its worst case no longer comes from putting weight Γ on the pre-onset minutes (Proposition 4, Appendix H.2), and finding it would require searching 2^L vertices over millions of minutes.
Instead, I use a sequential sensitivity model with parameter Λ ≥ 1. It reads the exact conditional null Q_ν as a Markov chain in time order and lets the odds of placing an alert at each minute, given the full history, take any value between 1/Λ and Λ times the null odds; minutes forced under the null (p0 = 0 or 1) stay forced, so every law in the model keeps the per-cell alert counts and the 240-minute cooldown.
I compute the worst-case tail probability sup P(T ≥ k) exactly by backward induction over (position, state, catches still needed), with the same state space as the notebook 34 exact law, to which the model reduces at Λ = 1.
For α = 0.05 and 0.01 I report the worst-case p on a Λ grid and Λ* (the largest Λ with worst-case p ≤ α), plus a Rosenbaum tilt for reference only.
This analysis is exploratory, and it does not change the preregistered confirmatory plan. I write its outputs to `sequential_sensitivity_v1`.
I record the pre-specified reading rules in `35_SPEC.json`: Λ* is a sensitivity bound for the retrospective conditional placement law Q_ν (with fixed monthly alert counts), not a model of real-time alert generation and not an effect size, and I do not read it as the same quantity as the Rosenbaum Γ*.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
OUT = SPOT_BASE / 'sequential_sensitivity_v1'; OUT.mkdir(parents=True, exist_ok=True)
KERNEL_REV = 'nb35_preflight_audit_20261102_r2'   # kernel revision label; included in SPEC
MAX_DP_MEMORY_FRACTION = 0.70                  # cap predicted peak working memory as a fraction of available RAM
MAX_ESTIMATED_WALL_HOURS = 12.0                 # estimated ~20 passes

def preflight_dp_memory(P, D, label):
    # _seq_backward_fast stores (R, D, Smax) V plus (R, Smax) Z and (L,D) Vfull and (L,) Zfull.
    # _backward stores (R,D,Smax) ring and (L,D) Gfull. 1.5x margin for allocations/compilation.
    smax = int(P['S'].max()); R = int(P['R']); L = int(P['L'])
    raw_gib = 8.0 * (R * (D + 1) * smax + L * (D + 2) + 2 * smax * D) / (1024 ** 3)
    est_gib = 1.5 * raw_gib
    try:
        import psutil
        avail_gib = psutil.virtual_memory().available / (1024 ** 3)
    except ImportError:
        try:                                            # fallback: Linux /proc/meminfo (kB)
            _mi = dict(l.split(':', 1) for l in open('/proc/meminfo'))
            avail_gib = float(_mi['MemAvailable'].split()[0]) / (1024 ** 2)
        except Exception:
            avail_gib = float('nan')
    print(f'Memory estimate {label}: D={D}; R={R}; largest radix state={smax:,}; '
          f'raw DP storage ≈ {raw_gib:.2f} GiB; estimated peak ≈ {est_gib:.2f} GiB; available RAM ≈ {avail_gib:.2f} GiB')
    if not np.isfinite(avail_gib):
        raise RuntimeError('Cannot establish the available RAM; set up the memory check before running the DP.')
    if est_gib > MAX_DP_MEMORY_FRACTION * avail_gib:
        raise MemoryError(f'Predicted DP memory {est_gib:.2f} GiB exceeds '
                          f'{MAX_DP_MEMORY_FRACTION:.0%} of available {avail_gib:.2f} GiB. '
                          'Use a high-RAM runtime or reduce nonessential D.')
    return est_gib
M16 = SPOT_BASE / 'methods_definitions_v1'
M34 = SPOT_BASE / 'exact_conditional_null_v1'
DNAME, DCL = '4pct', 0.04; H_REF = 240
SEED = 20261103
LAM_GRID = (1.25, 1.5, 2.0, 3.0)                       # worst-case p at these values; then bisection for lambda* inside the bracket
ALPHAS_STAR = (0.05, 0.01); BISECT_TOL = 0.02          # lambda* to within 2% (ratio of the bracket)
GAMMA_GRID = (1.25, 1.5, 2.0, 3.0, 4.0); KCAP_TILT = 45
C_MIN = COOLDOWN

for _p in [M16 / 'METHODS_FROZEN.json', M16 / 'METHODS_SPEC.json', M16 / f'alarms_M0_VF_logit_{DNAME}.csv', M34 / '2_m0_exact_vs_budget.csv', M34 / '2b_m0_exact_law.csv']:
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 16 and 34 are required: {_p}')
FROZEN16 = json.loads((M16 / 'METHODS_FROZEN.json').read_text())
if sha256_file(M16 / 'METHODS_SPEC.json') != FROZEN16['spec_sha256']:
    raise RuntimeError('GUARD FAIL: notebook-16 spec differs')
REF34 = pd.read_csv(M34 / '2_m0_exact_vs_budget.csv').iloc[0]
UPSTREAM_HASHES = {
    'nb16_alerts': sha256_file(M16 / f'alarms_M0_VF_logit_{DNAME}.csv'),
    'nb34_summary': sha256_file(M34 / '2_m0_exact_vs_budget.csv'),
    'nb34_exact_law': sha256_file(M34 / '2b_m0_exact_law.csv'),
    'spot_source_manifest': sha256_file(_src),
    'spot_reference': sha256_file(REF_PATH),
    'spot_thresholds': sha256_file(THR_PATH),
}
LAW34 = pd.read_csv(M34 / '2b_m0_exact_law.csv')            # notebook-34 exact law of T (bins 0..K-1 and '>= K')

SPEC = {'kernel_revision': KERNEL_REV, 'upstream_sha256': UPSTREAM_HASHES,
        'title': 'Sequential sensitivity analysis of the development H1 result under the exact conditional alert null (exploratory; appendix)',
        'null_model': 'Q_nu of notebook 34: alert set uniform over sets on the notebook-32 placement grid with the observed counts per (calendar month, RV decile) cell and spacing >= 240 minutes',
        'statistic': 'pre-onset catches T of the frozen M0 alerts (T_obs from notebook 34)',
        'sensitivity_model': 'sequential: at every grid minute, given the full history, the odds of placing an alert may take any value in [odds0 / lambda, odds0 * lambda], '
                             'odds0 = null conditional odds under Q_nu; forced minutes stay forced; worst case over history-dependent choices',
        'computation': 'exact backward induction over (position, remaining counts of the month, catches still needed); lambda = 1 must reproduce the whole notebook-34 tail P(T >= r), r = 0..T_obs; worst-case p must be non-decreasing in lambda',
        'interpretation': 'conditional on the observed event path; the reference conditional odds come from the retrospective conditional placement law Q_nu (fixed monthly counts), so this is a sensitivity model for that law, not a model of real-time alert generation; lambda* is not an effect size',
        'outputs': {'grid': list(LAM_GRID), 'lambda_star_alphas': list(ALPHAS_STAR), 'bisection_tolerance': BISECT_TOL},
        'reference': {'rosenbaum_tilt': 'weights gamma on pre-onset minutes; a lower bound on the Rosenbaum worst case under a cooldown (Proposition B.1), so gamma* is an upper bound on the Rosenbaum sensitivity value; recorded for reference only and not comparable with lambda* (different models)',
                      'gamma_grid': list(GAMMA_GRID), 'law_upper_bin': KCAP_TILT},
        'readings': 'lambda* and gamma* belong to different models and are not read as the same number; development sample was inspected repeatedly, so all outputs are exploratory',
        'seed': SEED, 'notebook16_spec_sha256': FROZEN16['spec_sha256'], 'holdouts': 'not used', 'prospective_folder': 'not read',
        'status': 'exploratory; appendix; no confirmatory plan changed'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '35_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    if any(OUT.glob('*.csv')) or (OUT / 'SEQ_SENS_SUMMARY.json').exists():
        raise RuntimeError('SPEC or upstream files changed but earlier CSV/summary outputs exist; move them out before a new run')
    _sp.rename(OUT / f'35_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
if not _sp.exists() and (any(OUT.glob('*.csv')) or (OUT / 'SEQ_SENS_SUMMARY.json').exists()):
    raise RuntimeError('Prior results found without 35_SPEC.json; provenance cannot be verified')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/sequential_sensitivity_v1 | spec sha256 46a060b418a44d1b


## 0. Reproduction of M0 scores, alerts and events (same code as section 0 of notebook 34)

In [2]:
t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
EVP = VALID & in_period(T, EVAL_START, EVAL_END)
_lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']
E_ref = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
CAT = price_only_catalog(close, volume, px.index, H=H_REF, decline=DCL)
for c_ in TCOLS:
    CAT[c_] = pd.to_datetime(CAT[c_], utc=True)
CAT = CAT.sort_values('breach_utc').reset_index(drop=True)
if len(CAT) != len(E_ref) or any(not np.array_equal(CAT[c_].astype('int64').to_numpy(), E_ref[c_].astype('int64').to_numpy()) for c_ in TCOLS):
    raise RuntimeError('GUARD FAIL: H=240 catalog differs from the notebook-2 catalog')
EV = events_between(CAT, EVAL_START, EVAL_END).reset_index(drop=True)
Xl = F[['z_vol', 'tsr']].to_numpy(float); okl = train_mask(T, VALID) & np.isfinite(Xl).all(axis=1)
cfg0 = FROZEN16['configs'][f'M0_VF_logit|{DNAME}']
m0 = new_logit().fit(Xl[okl], label_from_catalog(T, CAT)[okl])
if not (np.allclose(m0[-1].coef_.ravel(), cfg0['coef'], rtol=1e-6, atol=1e-9) and np.isclose(m0[-1].intercept_[0], cfg0['intercept'], rtol=1e-6, atol=1e-9)):
    raise RuntimeError('GUARD FAIL: M0 refit differs from notebook 16')
TH0 = float(cfg0['threshold_2022'])
S0_FULL = np.full(len(T), np.nan); okf = EVP & np.isfinite(Xl).all(axis=1); S0_FULL[okf] = m0.predict_proba(Xl[okf])[:, 1]
A_M0 = np.sort(pd.to_datetime(pd.read_csv(M16 / f'alarms_M0_VF_logit_{DNAME}.csv').alarm_utc, utc=True).astype('int64').to_numpy())
if len(A_M0) != cfg0['dev_alerts']:
    raise RuntimeError('GUARD FAIL: notebook-16 M0 alarms changed')
_ok0 = EVP & np.isfinite(S0_FULL)
if not np.array_equal(emit_calendar_fast(T[_ok0], S0_FULL[_ok0] >= TH0), A_M0):
    raise RuntimeError('GUARD FAIL: re-running the frozen scheduler on the M0 score does not reproduce the notebook-16 alerts')
del Xl
print(f'PASS: M0 model, alerts and scheduler verified; {len(EV)} events in the development period ({time.time() - t_:.0f}s)')


def ev_arrays(ev):
    return dict(br=ev.breach_utc.astype('int64').to_numpy(), on=ev.onset_1pct_utc.astype('int64').to_numpy(), pk=ev.peak_utc.astype('int64').to_numpy(), n=len(ev))


def fast_counts(a, ea):
    a = np.sort(np.asarray(a, np.int64)); used = np.zeros(len(a), bool); pre = post = 0
    for k in range(ea['n']):
        b = ea['br'][k]; l_ = np.searchsorted(a, b - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, b - LEAD_MIN * NS, 'right')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True
                if a[j] < ea['on'][k]:
                    pre += 1
                else:
                    post += 1
                break
    return dict(pre=pre, post=post)


def make_space(Tp, RVp, start, end):
    okd = (Tp >= start.value) & (Tp < end.value) & np.isfinite(RVp)
    tt = Tp[okd]; dq = np.minimum((pd.Series(RVp[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    mo = cal_fields(tt)[1]; key = mo * 10 + dq; o2 = np.lexsort((tt, key))
    return dict(TT=tt, DD=dq, PK=tt[o2], KS=key[o2])


EA4 = ev_arrays(EV); SP = make_space(T, np.where(VALID, RV, np.nan), ts(EVAL_START), ts(EVAL_END))
mm0 = match_events(EV, A_M0); fc0 = fast_counts(A_M0, EA4)
if (int(mm0.pre1pct.sum()), int((~mm0.pre1pct).sum())) != (fc0['pre'], fc0['post']):
    raise RuntimeError('GUARD FAIL: fast matcher differs from match_events')
if fc0['pre'] != int(REF34.pre_observed):
    raise RuntimeError('GUARD FAIL: M0 pre-onset catches differ from notebook 34')
print('M0 pre-onset catches:', fc0['pre'], '| grid minutes:', len(SP['TT']))

PASS: M0 model, alerts and scheduler verified; 83 events in the development period (25s)
M0 pre-onset catches: 22 | grid minutes: 1750857


## 1. Kernel verification against full enumeration on small designs

For small random designs (1–3 months, 1–3 cells, cooldown 1–5, with weights), I enumerate all possible alert placements, compute the worst-case tail probability by following every history and taking the worst-case end of the Λ range at each minute, and compare the result with the kernel.
For Λ = 1 I also compare the kernel with the tail of the notebook 34 exact null.

In [3]:
"""Exact conditional null for budget-constrained alerts (notebook 34 core).
Q_nu: uniform over alert sets A ⊂ positions with spacing >= C (time units) and exact counts nu[month, class].
Positions are sorted times; each has a month index, a class, an indicator h (pre-onset minute) and a weight w (1 = uniform).
Statistic T(A) = sum_{t in A} h(t) (additive because the judging window is shorter than the cooldown; checked separately on the real data)."""
import numpy as np
from numba import njit

# ---------------------------------------------------------------- design preparation
def prepare(times, month, cls, h, nu, C, w=None):
    """Drop positions whose (month, class) has zero count (they can never hold an alert), build radix states per month.
    month: int array (any labels, increasing in time); cls: int in [0, K); nu: dict {(month_label, class): count}."""
    times = np.asarray(times, np.int64); month = np.asarray(month, np.int64); cls = np.asarray(cls, np.int64); h = np.asarray(h, np.int64)
    assert np.all(np.diff(times) > 0)
    w = np.ones(len(times)) if w is None else np.asarray(w, float)
    K = int(max(cls.max() + 1, max(b for _, b in nu) + 1)) if nu else int(cls.max() + 1)
    mlabels = sorted({m for (m, b), v in nu.items() if v > 0})
    mid = {m: k for k, m in enumerate(mlabels)}; M = len(mlabels)
    NU = np.zeros((M, K), np.int64)
    for (m, b), v in nu.items():
        if v > 0: NU[mid[m], b] = v
    ml = np.array(mlabels, np.int64); pos = np.searchsorted(ml, month); pos_c = np.clip(pos, 0, max(M - 1, 0))
    keep = (M > 0) & (pos < M) & (ml[pos_c] == month) if M else np.zeros(len(month), bool)
    if M: keep &= NU[pos_c, np.clip(cls, 0, K - 1)] > 0
    t = times[keep]; mo = pos_c[keep].astype(np.int64); c = cls[keep]; hh = h[keep]; ww = w[keep]
    if M and not np.all(np.diff(mo) >= 0): raise ValueError('months must be increasing in time')
    L = len(t)
    mstart = np.searchsorted(mo, np.arange(M), 'left'); mend = np.searchsorted(mo, np.arange(M), 'right')
    if np.any(mend <= mstart): raise ValueError('a month with a positive count has no admissible position')
    RAD = NU + 1; STR = np.zeros((M, K), np.int64); S = np.ones(M, np.int64); FULL = np.zeros(M, np.int64)
    for m in range(M):
        s = 1
        for b in range(K):
            if NU[m, b] > 0: STR[m, b] = s; s *= RAD[m, b]
        S[m] = s; FULL[m] = int((NU[m] * STR[m]).sum())
    st = STR[mo, c]; rad = RAD[mo, c]
    nx = np.searchsorted(t, t + C, 'left').astype(np.int64)
    R = int((nx - np.arange(L)).max()) + 1 if L else 1
    return dict(t=t, mon=mo, cls=c, h=hh, w=ww, st=st, rad=rad, nx=nx, S=S, FULL=FULL, mstart=mstart, mend=mend, M=M, L=L, R=R,
                NU=NU, keep=keep, mlabels=mlabels, n=int(NU.sum()))

# ---------------------------------------------------------------- backward pass
@njit(cache=False)
def _pairs(D, mode, hi):
    """(destination bin, source bin) pairs for multiplying by u^hi (mode 0, saturating) or (1 + x)^hi (mode 1, truncated)."""
    if hi == 0:
        P = np.empty((D, 2), np.int64)
        for k in range(D): P[k, 0] = k; P[k, 1] = k
        return P
    if mode == 0:
        P = np.empty((D, 2), np.int64)
        for k in range(D - 1): P[k, 0] = k + 1; P[k, 1] = k
        P[D - 1, 0] = D - 1; P[D - 1, 1] = D - 1
        return P
    P = np.empty((2 * D - 1, 2), np.int64)
    for k in range(D): P[k, 0] = k; P[k, 1] = k
    for k in range(D - 1): P[D + k, 0] = k + 1; P[D + k, 1] = k
    return P

@njit(cache=False)
def _backward(nx, mon, st, rad, h, w, S, FULL, D, mode, R):
    """Backward recursion F(i, r; u) over kept positions. mode 0: saturating counter of T (bins 0..D-2 exact, bin D-1 = '>= D-1');
    mode 1: truncated polynomial in x = u - 1 (coefficients Z, Z E[T], Z E[T(T-1)]/2, ...). Ring values share one running log scale e;
    Gfull[i] is the value at the full state of month(i) with its scale lcfull[i] (true value = Gfull * exp(lcfull))."""
    L = len(nx); M = len(S); Smax = 1
    for m in range(M):
        if S[m] > Smax: Smax = S[m]
    ring = np.zeros((R, D, Smax)); Gfull = np.zeros((L, D)); lcfull = np.full(L, -np.inf)
    P0 = _pairs(D, mode, 0); P1 = _pairs(D, mode, 1); e = 0.0; BIG = 1e200; LBIG = np.log(BIG)
    for i in range(L - 1, -1, -1):
        m = mon[i]; Sm = S[m]; cur = ring[i % R]
        # skip term
        if i + 1 < L and mon[i + 1] == m:
            src = ring[(i + 1) % R]
            for k in range(D):
                for s in range(Sm): cur[k, s] = src[k, s]
        else:
            for k in range(D):
                for s in range(Sm): cur[k, s] = 0.0
            if i + 1 < L:
                if lcfull[i + 1] > -np.inf:
                    f = np.exp(lcfull[i + 1] - e)
                    for k in range(D): cur[k, 0] = f * Gfull[i + 1, k]
            else:
                cur[0, 0] = np.exp(-e)
        # take term
        j = nx[i]; wi = w[i]; stb = st[i]; rb = rad[i]
        PP = P1 if h[i] == 1 else P0
        if j < L and mon[j] == m:
            src = ring[j % R]; block = stb * rb; nq = Sm // block
            for p in range(PP.shape[0]):
                kd = PP[p, 0]; ks = PP[p, 1]
                for q in range(nq):
                    base = q * block
                    for r in range(1, rb):
                        o = base + r * stb
                        for l in range(stb):
                            cur[kd, o + l] += wi * src[ks, o - stb + l]
        elif stb < Sm:
            ok = False; f = 0.0
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf:
                ok = True; f = wi * np.exp(lcfull[j] - e)
            elif j >= L and m == M - 1:
                ok = True; f = wi * np.exp(-e)
            if ok:
                for p in range(PP.shape[0]):
                    kd = PP[p, 0]; ks = PP[p, 1]
                    if j < L: cur[kd, stb] += f * Gfull[j, ks]
                    elif ks == 0: cur[kd, stb] += f
        # occasional rescale of the whole ring (values only grow as i decreases)
        if (i & 15) == 0 or (i + 1 < L and mon[i + 1] != m):
            mx = 0.0
            for k in range(D):
                for s in range(Sm):
                    if cur[k, s] > mx: mx = cur[k, s]
            if mx > BIG:
                ring *= 1.0 / BIG; e += LBIG
        f_ = FULL[m]
        for k in range(D): Gfull[i, k] = cur[k, f_]
        lcfull[i] = e
    return Gfull, lcfull

def exact_law(P, kcap):
    """Exact law of T under Q_nu: P(T = k) for k < kcap and P(T >= kcap) (last entry)."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], int(kcap) + 1, 0, P['R'])
    v = G[0]
    if not v.sum() > 0: raise ValueError('no admissible configuration: the counts cannot be placed under the cooldown')
    return v / v.sum(), lc[0] + np.log(v.sum())

def exact_moments(P):
    """Exact E[T] and Var[T] under Q_nu from the factorial-moment pass."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 3, 1, P['R'])
    z, f1, f2 = G[0]; mean = f1 / z; e2 = 2 * f2 / z           # E[T(T-1)] = 2 * coefficient of x^2
    return mean, e2 + mean - mean ** 2

def log_count(P):
    """log of the number (weighted) of admissible configurations."""
    G, lc = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 1, 0, P['R'])
    return lc[0] + np.log(G[0, 0])

# ---------------------------------------------------------------- exact sampler
@njit(cache=False)
def _rows(m, lo, hi, after, after_e, nx, mon, st, rad, w, S, mend, Gfull, lcfull):
    """Backward rows (D = 1, u = 1) of month m for positions [lo, hi), given the rows of [hi, hi + len(after)) (empty when hi is the month end).
    Returns rows for [lo, hi + len(after)); row r holds F(lo + r, .) * exp(-erow[r])."""
    b = mend[m]; Sm = S[m]; L = len(nx); M = len(S); nA = after.shape[0]; tot = hi - lo + nA
    T = np.zeros((tot, Sm)); erow = np.zeros(tot); BIG = 1e200; LBIG = np.log(BIG)
    for r in range(nA):
        erow[hi - lo + r] = after_e[r]
        for s in range(Sm): T[hi - lo + r, s] = after[r, s]
    if nA > 0: e = after_e[0]
    else:
        e = lcfull[b] if b < L else 0.0
        if e == -np.inf: e = 0.0
    for i in range(hi - 1, lo - 1, -1):
        il = i - lo
        if i + 1 < b:
            f = np.exp(erow[il + 1] - e)
            for s in range(Sm): T[il, s] = f * T[il + 1, s]
        elif i + 1 < L:
            if lcfull[i + 1] > -np.inf: T[il, 0] = np.exp(lcfull[i + 1] - e) * Gfull[i + 1, 0]
        else:
            T[il, 0] = np.exp(-e)
        j = nx[i]; stb = st[i]; rb = rad[i]; wi = w[i]
        if j < b:
            f = wi * np.exp(erow[j - lo] - e); block = stb * rb; nq = Sm // block; jl = j - lo
            for q in range(nq):
                base = q * block
                for r in range(1, rb):
                    o = base + r * stb
                    for l in range(stb):
                        T[il, o + l] += f * T[jl, o - stb + l]
        elif stb < Sm:
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf: T[il, stb] += wi * np.exp(lcfull[j] - e) * Gfull[j, 0]
            elif j >= L and m == M - 1: T[il, stb] += wi * np.exp(-e)
        mx = 0.0
        for s in range(Sm):
            if T[il, s] > mx: mx = T[il, s]
        if mx > BIG:
            for s in range(Sm): T[il, s] /= BIG
            e += LBIG
        erow[il] = e
    return T, erow

@njit(cache=False)
def _walk_seg(m, lo, hi, T, erow, nx, mon, st, rad, w, mend, FULL, Gfull, lcfull, cur_i, cur_s, out, nout, seed):
    """Advance every draw whose current position lies in [lo, hi) of month m until it leaves the segment."""
    np.random.seed(seed)
    b = mend[m]; L = len(nx); M = len(FULL)
    for d in range(len(cur_i)):
        i = cur_i[d]; s = cur_s[d]
        if i < lo or i >= hi: continue
        while i < hi:
            if s == 0:
                i = b; break
            il = i - lo; j = nx[i]; tv = 0.0; stb = st[i]
            if (s // stb) % rad[i] >= 1:
                if j < b:
                    tv = w[i] * T[j - lo, s - stb] * np.exp(erow[j - lo] - erow[il])
                elif s - stb == 0:
                    if j < L and mon[j] == m + 1: tv = w[i] * Gfull[j, 0] * np.exp(lcfull[j] - erow[il])
                    elif j >= L and m == M - 1: tv = w[i] * np.exp(-erow[il])
            sv = T[il + 1, s] * np.exp(erow[il + 1] - erow[il]) if i + 1 < b else 0.0
            p = tv / (tv + sv) if tv + sv > 0.0 else 0.0
            if np.random.random() < p:
                out[d, nout[d]] = i; nout[d] += 1; s -= stb; i = j
                if j >= b: break
            else:
                i += 1
        cur_i[d] = i
        if i >= b: cur_s[d] = FULL[mon[i]] if i < L else 0
        else: cur_s[d] = s

def sample_exact(P, ndraw, seed=0, seg=4000):
    """ndraw independent exact draws from Q_nu; returns an (ndraw, n) array of kept-position indices (sorted per draw).
    Each month is processed in segments of `seg` positions: one backward pass stores only the rows just after each segment boundary,
    and each segment is recomputed from its checkpoint when the draws walk through it (memory about (seg + R) x states per month)."""
    G, lcf = _backward(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], 1, 0, P['R'])
    n = P['n']; R = P['R']; out = np.full((ndraw, n), -1, np.int64); nout = np.zeros(ndraw, np.int64)
    cur_i = np.zeros(ndraw, np.int64); cur_s = np.full(ndraw, P['FULL'][0], np.int64)
    args = (P['nx'], P['mon'], P['st'], P['rad'], P['w'], P['S'], P['mend'], G, lcf)
    for m in range(P['M']):
        a, b = int(P['mstart'][m]), int(P['mend'][m]); Sm = int(P['S'][m])
        bounds = list(range(a, b, seg)) + [b]; nseg = len(bounds) - 1
        ck = [None] * nseg; after = np.zeros((0, Sm)); after_e = np.zeros(0)
        for k in range(nseg - 1, -1, -1):
            lo, hi = bounds[k], bounds[k + 1]
            ck[k] = (after, after_e)
            Tm, er = _rows(m, lo, hi, after, after_e, *args)
            nA = min(R, b - lo); after = Tm[:nA].copy(); after_e = er[:nA].copy()
        for k in range(nseg):
            lo, hi = bounds[k], bounds[k + 1]
            Tm, er = _rows(m, lo, hi, ck[k][0], ck[k][1], *args)
            _walk_seg(m, lo, hi, Tm, er, P['nx'], P['mon'], P['st'], P['rad'], P['w'], P['mend'], P['FULL'], G, lcf,
                      cur_i, cur_s, out, nout, (seed * 1000003 + m) * 1009 + k)
    if np.any(nout != n): raise RuntimeError('sampler did not place every alert')
    return out

def check_draws(P, idx, C):
    """Constraint audit of exact draws: counts per (month, class) and spacing."""
    t = P['t'][idx]; bad_space = int((np.diff(t, axis=1) < C).sum())
    bad_count = 0
    for d in range(idx.shape[0]):
        cnt = np.zeros_like(P['NU']); np.add.at(cnt, (P['mon'][idx[d]], P['cls'][idx[d]]), 1)
        bad_count += int(np.any(cnt != P['NU']))
    return bad_space, bad_count
"""Sequential sensitivity analysis for the exact conditional alert null (notebook 35 core).

Null Q_nu: uniform (or w-weighted) over alert sets with exact counts per (month, class) and spacing >= C (exactnull.py).
Read sequentially in time, Q_nu is a Markov chain: at kept position i in state s (remaining counts of the month),
an alert is placed with null probability p0(i, s) = w_i Z(take) / (w_i Z(take) + Z(skip)).

Sequential sensitivity model with parameter Lam >= 1: at every position, given the whole history, the odds of placing an
alert may be any value in [odds0 / Lam, odds0 * Lam]; positions where the null is forced (p0 = 0 or 1) stay forced, so every
law in the model satisfies the counts and the spacing. The worst-case tail sup P(T >= k) over all (history-dependent)
choices is a finite-horizon control problem; backward induction over (position, state, catches still needed) solves it
exactly, with the same state space and cost as the exact law of T.
"""
import numpy as np
from numba import njit, prange


@njit(cache=False)
def _seq_backward(nx, mon, st, rad, h, w, S, FULL, D, R, lam):
    """V[r] = sup P(future catches >= r), r = 0..D-1, from the start of the design; also the null tail for checking (lam = 1)."""
    L = len(nx); M = len(S); Smax = 1
    for m in range(M):
        if S[m] > Smax: Smax = S[m]
    Zr = np.zeros((R, Smax)); Vr = np.zeros((R, D, Smax))
    Zfull = np.zeros(L); lcfull = np.full(L, -np.inf); Vfull = np.zeros((L, D))
    e = 0.0; BIG = 1e200; LBIG = np.log(BIG)
    zs = np.zeros(Smax); zt = np.zeros(Smax); vs = np.zeros((D, Smax)); vt = np.zeros((D, Smax))
    for i in range(L - 1, -1, -1):
        m = mon[i]; Sm = S[m]; row = i % R
        # ---- skip branch
        if i + 1 < L and mon[i + 1] == m:
            nr = (i + 1) % R
            for s in range(Sm):
                zs[s] = Zr[nr, s]
                for r in range(D): vs[r, s] = Vr[nr, r, s]
        else:
            for s in range(Sm):
                zs[s] = 0.0
                for r in range(D): vs[r, s] = 0.0
            if i + 1 < L:
                if lcfull[i + 1] > -np.inf and Zfull[i + 1] > 0.0:
                    zs[0] = np.exp(lcfull[i + 1] - e) * Zfull[i + 1]
                    for r in range(D): vs[r, 0] = Vfull[i + 1, r]
            else:
                zs[0] = np.exp(-e)
                vs[0, 0] = 1.0
        # ---- take branch
        j = nx[i]; wi = w[i]; stb = st[i]; rb = rad[i]; hi = h[i]
        for s in range(Sm):
            zt[s] = 0.0
            for r in range(D): vt[r, s] = 0.0
        if j < L and mon[j] == m:
            jr = j % R
            for s in range(Sm):
                if (s // stb) % rb >= 1:
                    zt[s] = wi * Zr[jr, s - stb]
                    for r in range(D):
                        rr = r - hi
                        if rr < 0: rr = 0
                        vt[r, s] = Vr[jr, rr, s - stb]
        elif stb < Sm:
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf and Zfull[j] > 0.0:
                zt[stb] = wi * np.exp(lcfull[j] - e) * Zfull[j]
                for r in range(D):
                    rr = r - hi
                    if rr < 0: rr = 0
                    vt[r, stb] = Vfull[j, rr]
            elif j >= L and m == M - 1:
                zt[stb] = wi * np.exp(-e)
                for r in range(D):
                    rr = r - hi
                    if rr < 0: rr = 0
                    vt[r, stb] = 1.0 if rr == 0 else 0.0
        # ---- combine: null count and worst-case value
        mx = 0.0
        for s in range(Sm):
            z = zs[s] + zt[s]; Zr[row, s] = z
            if z > mx: mx = z
            if zt[s] > 0.0 and zs[s] > 0.0:
                o0 = zt[s] / zs[s]; olo = o0 / lam; ohi = o0 * lam
                plo = olo / (1.0 + olo); phi = ohi / (1.0 + ohi)
                for r in range(D):
                    a = plo * vt[r, s] + (1.0 - plo) * vs[r, s]
                    b = phi * vt[r, s] + (1.0 - phi) * vs[r, s]
                    Vr[row, r, s] = a if a > b else b
            elif zt[s] > 0.0:
                for r in range(D): Vr[row, r, s] = vt[r, s]
            else:
                for r in range(D): Vr[row, r, s] = vs[r, s]
        if mx > BIG:
            for q in range(R):
                for s in range(Smax): Zr[q, s] /= BIG
            e += LBIG
        f_ = FULL[m]; Zfull[i] = Zr[row, f_]; lcfull[i] = e
        for r in range(D): Vfull[i, r] = Vr[row, r, f_]
    return Vfull[0].copy(), Zfull[0], lcfull[0]

@njit(cache=False, parallel=True)
def _seq_backward_fast(nx, mon, st, rad, h, w, S, FULL, D, R, lam):
    """Same result as _seq_backward; one sweep per state, states in parallel."""
    L = len(nx); M = len(S); Smax = 1
    for m in range(M):
        if S[m] > Smax: Smax = S[m]
    Zr = np.zeros((R, Smax)); Vr = np.zeros((R, D, Smax))
    Zfull = np.zeros(L); lcfull = np.full(L, -np.inf); Vfull = np.zeros((L, D))
    e = 0.0; BIG = 1e200; LBIG = np.log(BIG)
    endv = np.zeros(D); endv[0] = 1.0
    for i in range(L - 1, -1, -1):
        m = mon[i]; Sm = S[m]; row = i % R
        j = nx[i]; wi = w[i]; stb = st[i]; rb = rad[i]; hi = h[i]
        same_next = (i + 1 < L) and (mon[i + 1] == m)
        nr = (i + 1) % R
        # month-end skip value (state 0 only)
        zs0 = 0.0; vs0 = endv
        if not same_next:
            if i + 1 < L:
                if lcfull[i + 1] > -np.inf and Zfull[i + 1] > 0.0:
                    zs0 = np.exp(lcfull[i + 1] - e) * Zfull[i + 1]; vs0 = Vfull[i + 1]
            else:
                zs0 = np.exp(-e)
        # take mode: 0 none, 1 same month (ring row j), 2 cross-month / end (state stb only)
        tmode = 0; zt_x = 0.0; vt_x = endv; jr = j % R
        if j < L and mon[j] == m:
            tmode = 1
        elif stb < Sm:
            if j < L and mon[j] == m + 1 and lcfull[j] > -np.inf and Zfull[j] > 0.0:
                tmode = 2; zt_x = wi * np.exp(lcfull[j] - e) * Zfull[j]; vt_x = Vfull[j]
            elif j >= L and m == M - 1:
                tmode = 2; zt_x = wi * np.exp(-e)
        for s in prange(Sm):
            if same_next:
                zs = Zr[nr, s]
            else:
                zs = zs0 if s == 0 else 0.0
            zt = 0.0
            if tmode == 1:
                if (s // stb) % rb >= 1: zt = wi * Zr[jr, s - stb]
            elif tmode == 2:
                if s == stb: zt = zt_x
            Zr[row, s] = zs + zt
            if zt > 0.0 and zs > 0.0:
                o0 = zt / zs; olo = o0 / lam; ohi = o0 * lam
                plo = olo / (1.0 + olo); phi = ohi / (1.0 + ohi)
                for r in range(D):
                    rr = r - hi
                    if rr < 0: rr = 0
                    vt = Vr[jr, rr, s - stb] if tmode == 1 else vt_x[rr]
                    vs = Vr[nr, r, s] if same_next else vs0[r]
                    a = plo * vt + (1.0 - plo) * vs; b = phi * vt + (1.0 - phi) * vs
                    Vr[row, r, s] = a if a > b else b
            elif zt > 0.0:
                for r in range(D):
                    rr = r - hi
                    if rr < 0: rr = 0
                    Vr[row, r, s] = Vr[jr, rr, s - stb] if tmode == 1 else vt_x[rr]
            elif zs > 0.0:
                for r in range(D):
                    Vr[row, r, s] = Vr[nr, r, s] if same_next else vs0[r]
            else:
                for r in range(D): Vr[row, r, s] = 0.0
        mx = 0.0
        for s in range(Sm):
            if Zr[row, s] > mx: mx = Zr[row, s]
        if mx > BIG:
            for q in range(R):
                for s in range(Smax): Zr[q, s] /= BIG
            e += LBIG
        f_ = FULL[m]; Zfull[i] = Zr[row, f_]; lcfull[i] = e
        for r in range(D): Vfull[i, r] = Vr[row, r, f_]
    return Vfull[0].copy(), Zfull[0], lcfull[0]


def worst_tail(P, k, lam, fast=True):
    """sup over the sequential sensitivity model with parameter lam of P(T >= r), for r = 0..k (array of length k + 1).
    worst_tail(P, k, 1.0) equals the exact null tail."""
    kern = _seq_backward_fast if fast else _seq_backward
    V, z0, _ = kern(P['nx'], P['mon'], P['st'], P['rad'], P['h'], P['w'], P['S'], P['FULL'], int(k) + 1, P['R'], float(lam))
    if not z0 > 0: raise ValueError('no admissible configuration')
    return V


def tilt_weights(P, gamma):
    """Rosenbaum-type tilt for reference: weight gamma on pre-onset positions (h = 1), 1 elsewhere (not the worst case under a
    cooldown; Proposition 4). Use with exactnull.exact_law on a copy of P."""
    Q = dict(P); Q['w'] = np.where(P['h'] == 1, float(gamma), 1.0) * P['w']
    return Q


def lam_star(P, k, alpha=0.05, lo=1.0, hi=8.0, tol=0.01, log=None):
    """Largest lam with sup P(T >= k) <= alpha (bisection on log lam; the worst-case tail is non-decreasing in lam)."""
    f = lambda L_: worst_tail(P, k, L_)[k]
    if f(lo) > alpha: return lo, [(lo, f(lo))]
    hist = []
    while f(hi) <= alpha:
        hist.append((hi, f(hi))); lo = hi; hi *= 2
        if hi > 1e4: return np.inf, hist
    while hi / lo > 1 + tol:
        mid = np.sqrt(lo * hi); v = f(mid); hist.append((mid, v))
        if log: log(mid, v)
        if v <= alpha: lo = mid
        else: hi = mid
    return lo, hist
import itertools, random
from functools import lru_cache


def _enumerate(times, month, cls, nu, C):
    n = sum(nu.values()); out = []
    for comb in itertools.combinations(range(len(times)), n):
        if any(times[comb[k + 1]] - times[comb[k]] < C for k in range(n - 1)):
            continue
        cnt = {}
        for i in comb:
            cnt[(month[i], cls[i])] = cnt.get((month[i], cls[i]), 0) + 1
        if all(cnt.get(c, 0) == v for c, v in nu.items() if v > 0) and all(nu.get(c, 0) == v for c, v in cnt.items()):
            out.append(comb)
    return out


def _brute_worst(A, w, h, k, lam, L):
    sets = [frozenset(a) for a in A]; wt = {a: float(np.prod([w[t] for t in a])) for a in sets}

    @lru_cache(maxsize=None)
    def V(i, H):
        if i == L:
            return 1.0 if sum(h[t] for t in H) >= k else 0.0
        comp = [a for a in sets if frozenset(t for t in a if t < i) == H]
        zt = sum(wt[a] for a in comp if i in a); zs = sum(wt[a] for a in comp if i not in a)
        if zt == 0:
            return V(i + 1, H)
        if zs == 0:
            return V(i + 1, H | {i})
        o0 = zt / zs; vt = V(i + 1, H | {i}); vs = V(i + 1, H)
        return max(o / (1 + o) * vt + 1 / (1 + o) * vs for o in (o0 / lam, o0 * lam))
    return V(0, frozenset())


random.seed(SEED); _r = np.random.default_rng(SEED)
vrows = []; t_ = time.time()
for trial in range(600):
    L_ = random.randint(5, 11); times = np.cumsum([random.choice([1, 1, 1, 2, 3]) for _ in range(L_)])
    M_ = random.randint(1, 3); K_ = random.randint(1, 3); C_ = random.randint(1, 5)
    month = np.sort(_r.integers(0, M_, L_)) * 7 + 3; cls = _r.integers(0, K_, L_); h = (_r.random(L_) < 0.4).astype(int)
    w = np.where(_r.random(L_) < 0.5, 1.0, _r.choice([0.5, 2.0], L_)) if trial % 4 == 0 else np.ones(L_)
    nu = {c: random.randint(0, 2) for c in sorted(set(zip(month.tolist(), cls.tolist())))}
    if sum(nu.values()) == 0:
        continue
    try:
        P_ = prepare(times, month, cls, h, nu, C_, w)
    except ValueError:
        continue
    A_ = _enumerate(times.tolist(), month.tolist(), cls.tolist(), nu, C_)
    if not A_:
        continue
    kmax = int(max(h[list(a)].sum() for a in A_)) + 1; lam_ = random.choice([1.3, 2.0, 3.5])
    V_ = worst_tail(P_, kmax, lam_)
    ref = np.array([_brute_worst(A_, w, h, kk, lam_, L_) for kk in range(kmax + 1)])
    law_, _ = exact_law(P_, kmax); tail_ = np.cumsum(law_[::-1])[::-1]
    vrows.append(dict(trial=trial, configs=len(A_), cooldown=C_, lam=lam_, err=float(np.abs(V_ - ref).max()),
                      err_lam1=float(np.abs(worst_tail(P_, kmax, 1.0) - tail_).max())))
VR = pd.DataFrame(vrows); VR.to_csv(OUT / '1_verification_small.csv', index=False)
print(f'{len(VR)} small designs: max error vs full recursion {VR.err.max():.1e}; lambda = 1 vs exact null tail {VR.err_lam1.max():.1e} ({time.time() - t_:.0f}s)')
if not (VR.err.max() < 1e-10 and VR.err_lam1.max() < 1e-10):
    raise RuntimeError('The sequential worst-case kernel failed the small-problem verification')
print('PASS: worst-case kernel agrees with the full recursion')

307 small designs: max error vs full recursion 8.9e-16; lambda = 1 vs exact null tail 2.2e-16 (10s)
PASS: worst-case kernel agrees with the full recursion


## 2. Real design and Λ = 1 check

I build the grid, cells and pre-onset minutes h(t) with the same code as section 2 of notebook 34 and compare the worst-case p at Λ = 1 with the notebook 34 exact p.

In [4]:
TT = SP['TT']; GRID_MIN = (TT // NS).astype(np.int64); GRID_MON = cal_fields(TT)[1]; GRID_DEC = SP['DD'].astype(np.int64)
lo_w = EA4['br'] - LEAD_MAX * NS; hi_w = EA4['br'] - LEAD_MIN * NS
if np.any(lo_w[1:] <= hi_w[:-1]) or (LEAD_MAX - LEAD_MIN) >= COOLDOWN:
    raise RuntimeError('GUARD FAIL: T is not additive')
H_GRID = np.zeros(len(TT), np.int64)
for k in range(EA4['n']):
    l_ = np.searchsorted(TT, lo_w[k], 'left'); r_ = np.searchsorted(TT, hi_w[k], 'right')
    H_GRID[l_:r_] = (TT[l_:r_] < EA4['on'][k]).astype(np.int64)


def cells_of(a):
    a = np.asarray(a, np.int64); j = np.clip(np.searchsorted(TT, a, 'right') - 1, 0, len(TT) - 1)
    return cal_fields(a)[1], GRID_DEC[j], TT[j] == a


def design_for(a):
    mo, dq, on_grid = cells_of(a)
    nu = {}
    for m_, d_ in zip(mo.tolist(), dq.tolist()):
        nu[(m_, d_)] = nu.get((m_, d_), 0) + 1
    return prepare(GRID_MIN, GRID_MON, GRID_DEC, H_GRID, nu, C_MIN), int((~on_grid).sum())


def t_additive(a):
    a = np.asarray(a, np.int64); k = np.searchsorted(lo_w, a, 'right') - 1; ok = k >= 0
    kk = np.clip(k, 0, len(lo_w) - 1)
    return int((ok & (a <= hi_w[kk]) & (a < EA4['on'][kk])).sum())


P0, off0 = design_for(A_M0); T0 = fc0['pre']
if off0:
    raise RuntimeError(f'{off0} observed M0 alerts are off the Q candidate grid; exact conditional p is not defined for that observed placement')
if P0['n'] != len(A_M0) or not np.all(np.isin(A_M0 // NS, P0['t'])):
    raise RuntimeError('Observed M0 alerts do not lie in the prepared Q support')
if len(A_M0) > 1 and np.any(np.diff(A_M0) < COOLDOWN * NS):
    raise RuntimeError('Observed M0 alerts violate the declared cooldown')
if t_additive(A_M0) != T0:
    raise RuntimeError('GUARD FAIL: sum of h over M0 alerts differs from fast_counts pre')
_mem = (P0['R'] * (T0 + 1) * int(P0['S'].max()) + P0['L'] * (T0 + 1)) * 8 / 2 ** 30
print(f'M0 design: {P0["n"]} alerts, {P0["M"]} months, positions {P0["L"]}, state sizes median {int(np.median(P0["S"]))} / max {int(P0["S"].max())}, '
      f'alerts off the grid {off0}; memory per worst-case pass about {_mem:.1f} GiB')
preflight_dp_memory(P0, T0 + 1, 'lambda=1 / sequential')
t_ = time.time()
V1 = worst_tail(P0, T0, 1.0); SEC_PASS = time.time() - t_
rel = abs(V1[T0] - float(REF34.exact_p)) / float(REF34.exact_p)
_tail34 = LAW34.set_index('k')['tail']
if int(_tail34.index.max()) <= T0:
    raise RuntimeError('notebook-34 law has no bin above T_obs')
rel_all = max(abs(V1[r] - float(_tail34[r])) / float(_tail34[r]) for r in range(T0 + 1) if float(_tail34[r]) > 0)
pd.DataFrame([dict(r=r, tail_lambda1=float(V1[r]), tail_nb34=float(_tail34[r])) for r in range(T0 + 1)]).to_csv(OUT / '2b_lambda1_tail_check.csv', index=False)
pd.DataFrame([dict(T_obs=T0, p_lambda1=V1[T0], p_exact_nb34=float(REF34.exact_p), rel_diff=rel, max_rel_diff_all_r=rel_all, seconds=SEC_PASS)]).to_csv(OUT / '2_lambda1_check.csv', index=False)
print(f'lambda = 1: worst-case p {V1[T0]:.4e} vs notebook-34 exact p {float(REF34.exact_p):.4e} (relative difference {rel:.1e}); '
      f'whole tail P(T >= r), r = 0..{T0}: largest relative difference {rel_all:.1e}; one pass {SEC_PASS / 60:.1f} min')
if not (rel < 1e-6 and rel_all < 1e-6):
    raise RuntimeError('Lambda = 1 does not reproduce the notebook-34 exact law of T')
_forecast_passes = len(LAM_GRID) + 2 * 8 + 1  # rough, not an upper bound; depends on threshold crossings
_forecast_hours = _forecast_passes * SEC_PASS / 3600
print(f'Time estimate: {SEC_PASS / 60:.2f} min per lambda=1 pass, ~{_forecast_passes} sequential passes ~{_forecast_hours:.1f} hours (estimate only)')
if _forecast_hours > MAX_ESTIMATED_WALL_HOURS:
    raise RuntimeError(f'Projected sequential calculation {_forecast_hours:.1f} h exceeds configured limit {MAX_ESTIMATED_WALL_HOURS:.1f} h; change limit explicitly only after reviewing runtime capacity')

M0 design: 800 alerts, 40 months, positions 1414298, state sizes median 5040 / max 25920, alerts off the grid 0; memory per worst-case pass about 1.3 GiB
Memory estimate lambda=1 / sequential: D=23; R=241; largest radix state=25,920; raw DP storage ≈ 1.39 GiB; estimated peak ≈ 2.08 GiB; available RAM ≈ 9.89 GiB
lambda = 1: worst-case p 7.3896e-08 vs notebook-34 exact p 7.3896e-08 (relative difference 4.3e-15); whole tail P(T >= r), r = 0..22: largest relative difference 5.6e-13; one pass 25.8 min
Time estimate: 25.76 min per lambda=1 pass, ~21 sequential passes ~9.0 hours (estimate only)


## 3. Worst-case p and Λ*

On a Λ grid I compute the worst-case p = sup P(T ≥ 22), and for α = 0.05 and 0.01 I find the largest Λ with worst-case p ≤ α by bisection (log scale, bracket ratio within 2%).
Because the models are nested in Λ, the worst-case p is non-decreasing in Λ; I save every Λ I evaluate to `3_worst_case.csv`.

In [5]:
wpath = OUT / '3_worst_case.csv'
WC = pd.read_csv(wpath) if wpath.exists() else pd.DataFrame(columns=['lam', 'p_worst', 'seconds', 'tail_r'])
done = {round(float(x), 10): float(p) for x, p in zip(WC.lam, WC.p_worst)}


def p_worst(lam):
    key = round(float(lam), 10)
    if key in done:
        return done[key]
    t0_ = time.time(); V_ = worst_tail(P0, T0, lam); sec = time.time() - t0_
    done[key] = float(V_[T0])
    row = pd.DataFrame([dict(lam=key, p_worst=float(V_[T0]), seconds=sec, tail_r=json.dumps([float(x) for x in V_]))])
    row.to_csv(wpath, mode='a', header=not wpath.exists(), index=False)
    print(f'  lambda {lam:.4f}: worst-case p {V_[T0]:.4e} ({sec / 60:.1f} min)', flush=True)
    return done[key]


for lam in LAM_GRID:
    p_worst(lam)
STAR = []
for a_ in ALPHAS_STAR:
    grid = sorted([1.0] + list(LAM_GRID)); ps = [p_worst(x) if x > 1 else float(V1[T0]) for x in grid]
    if ps[0] > a_:
        STAR.append(dict(alpha=a_, lambda_star=1.0, bracket_lo=1.0, bracket_hi=1.0, note='not significant at lambda = 1')); continue
    hi_ = next((x for x, p in zip(grid, ps) if p > a_), None)
    while hi_ is None:
        x = grid[-1] * 2; grid.append(x)
        if p_worst(x) > a_:
            hi_ = x
        if x > 64:
            break
    if hi_ is None:
        STAR.append(dict(alpha=a_, lambda_star=np.nan, bracket_lo=grid[-1], bracket_hi=np.nan,
                         note='right-censored: still significant at maximum tested lambda; report lambda_star > bracket_lo, not infinity')); continue
    lo_ = max(x for x in grid if x < hi_ and (x == 1.0 or p_worst(x) <= a_))
    while hi_ / lo_ > 1 + BISECT_TOL:
        mid = float(np.sqrt(lo_ * hi_))
        if p_worst(mid) <= a_:
            lo_ = mid
        else:
            hi_ = mid
    STAR.append(dict(alpha=a_, lambda_star=lo_, bracket_lo=lo_, bracket_hi=hi_, note='largest lambda with worst-case p <= alpha (within the bracket)'))
STAR = pd.DataFrame(STAR); STAR.to_csv(OUT / '3b_lambda_star.csv', index=False)
WC = pd.read_csv(wpath).sort_values('lam'); display(WC[['lam', 'p_worst', 'seconds']]); display(STAR)
_pw = np.r_[float(V1[T0]), WC.p_worst.to_numpy(float)]
MONO_OK = bool(np.all(np.diff(_pw) >= -(1e-12 + 1e-9 * np.abs(_pw[:-1]))))
_tails = np.array([json.loads(x) for x in WC.tail_r]); _ok_r = bool(np.all(np.diff(_tails, axis=1) <= 1e-12)) and bool(np.all((_tails >= -1e-12) & (_tails <= 1 + 1e-12)))
print(f'worst-case p non-decreasing in lambda over {len(_pw)} values (lambda = 1 included): {MONO_OK}; '
      f'every worst-case tail lies in [0, 1] and is non-increasing in r: {_ok_r}')
if not (MONO_OK and _ok_r):
    raise RuntimeError('Worst-case p is not monotone in lambda (or a tail is malformed), so it cannot be interpreted')

  lambda 1.6119: worst-case p 8.4929e-03 (25.5 min)
  lambda 1.6709: worst-case p 1.5503e-02 (26.5 min)
  lambda 1.6411: worst-case p 1.1537e-02 (27.2 min)


,lam,p_worst,seconds
0,1.250000,0.000037,631.496204
1,1.500000,0.002243,557.940291
11,1.611855,0.008493,1529.216005
8,1.611855,0.008493,588.648507
13,1.641098,0.011537,1631.549859
10,1.641098,0.011537,558.216052
12,1.670872,0.015503,1591.539379
9,1.670872,0.015503,563.618361
4,1.732051,0.027091,549.974379
6,1.795469,0.045301,559.038785


,alpha,lambda_star,bracket_lo,bracket_hi,note
0,0.05,1.795469,1.795469,1.828044,largest lambda with worst-case p <= alpha (wit...
1,0.01,1.611855,1.611855,1.641098,largest lambda with worst-case p <= alpha (wit...


worst-case p non-decreasing in lambda over 15 values (lambda = 1 included): True; every worst-case tail lies in [0, 1] and is non-increasing in r: True


## 4. Reference: Rosenbaum tilt

With weight Γ on pre-onset minutes, a placement A has weight Γ^T(A), so the tilted law is the null law multiplied by Γ^k and renormalized; I compute the null law once (top bin 45, recording the tilted mass in the last bin), and it gives the tail for every Γ.
Under a cooldown this value is a lower bound on the Rosenbaum worst case (Proposition 4), so the Γ* I find here is an upper bound on the Rosenbaum sensitivity value.

In [6]:
import gc; gc.collect()
if KCAP_TILT < T0:
    raise RuntimeError('Reference tilt bin is below observed T, so the lower-bound interpretation breaks')
try:
    preflight_dp_memory(P0, KCAP_TILT + 1, 'Rosenbaum reference (truncated tail)'); TILT_RUN = True
except MemoryError as e_:
    TILT_RUN = False
    print(f'SKIP section 4 (reference only; lambda* in section 3 is unaffected): {e_}')

if TILT_RUN:
    t_ = time.time()
    LAW60, _ = exact_law(P0, KCAP_TILT); kk_ = np.arange(len(LAW60))
    print(f'null law to {KCAP_TILT}: null mass in the last bin (T >= {KCAP_TILT}) {LAW60[-1]:.1e} ({time.time() - t_:.0f}s)')


    def tilt_p(g):
        lw = np.log(np.maximum(LAW60, 1e-300)) + kk_ * np.log(g); lw -= lw.max(); wv = np.exp(lw) * (LAW60 > 0)
        return float(wv[T0:].sum() / wv.sum())


    # For gamma>1, folding T>=KCAP_TILT into the KCAP_TILT bin understates the tilted upper tail.
    # Therefore this is a lower bound on the specified Rosenbaum tilt tail, not an exact tilted p-value.
    TR = pd.DataFrame([dict(gamma=g, p_tilt_lower_bound=tilt_p(g)) for g in (1.0,) + GAMMA_GRID])
    GSTAR = []
    for a_ in ALPHAS_STAR:
        lo_, hi_ = 1.0, 2.0
        while tilt_p(hi_) <= a_ and hi_ < 1e3:
            lo_, hi_ = hi_, hi_ * 2
        if tilt_p(hi_) <= a_:
            GSTAR.append(dict(alpha=a_, gamma_star_upper_bound_from_truncated_tilt=np.nan,
                              tilted_last_bin_mass=np.nan,
                              note=f'not crossed for gamma <= {hi_:.6g}; no finite gamma upper bound established'))
            continue
        for _ in range(60):
            mid = float(np.sqrt(lo_ * hi_))
            lo_, hi_ = (mid, hi_) if tilt_p(mid) <= a_ else (lo_, mid)
        lw = np.log(np.maximum(LAW60, 1e-300)) + kk_ * np.log(lo_); wv = np.exp(lw - lw.max()) * (LAW60 > 0)
        GSTAR.append(dict(alpha=a_, gamma_star_upper_bound_from_truncated_tilt=lo_,
                          tilted_last_bin_mass=float(wv[-1] / wv.sum())))
    TR.to_csv(OUT / '4_rosenbaum_tilt.csv', index=False); GSTAR = pd.DataFrame(GSTAR); GSTAR.to_csv(OUT / '4b_gamma_star_tilt.csv', index=False)
    display(TR); display(GSTAR)
    if GSTAR.tilted_last_bin_mass.max() > 1e-3:
        print('Warning: the Rosenbaum reference bound may be very loose, because the saturated last bin carries substantial tilted probability; it is not a precise gamma threshold.')
else:
    LAW60 = np.array([np.nan])
    TR = pd.DataFrame([dict(gamma=np.nan, p_tilt_lower_bound=np.nan, note='skipped: memory check')])
    GSTAR = pd.DataFrame([dict(alpha=a_, gamma_star_upper_bound_from_truncated_tilt=np.nan, tilted_last_bin_mass=np.nan, note='skipped: memory check') for a_ in ALPHAS_STAR])
    TR.to_csv(OUT / '4_rosenbaum_tilt.csv', index=False); GSTAR.to_csv(OUT / '4b_gamma_star_tilt.csv', index=False)
    display(TR); display(GSTAR)


Memory estimate Rosenbaum reference (truncated tail): D=46; R=241; largest radix state=25,920; raw DP storage ≈ 2.71 GiB; estimated peak ≈ 4.07 GiB; available RAM ≈ 9.68 GiB
null law to 45: null mass in the last bin (T >= 45) 1.2e-32 (1922s)


,gamma,p_tilt_lower_bound
0,1.00,7.389591e-08
1,1.25,2.048641e-06
2,1.50,2.469746e-05
3,2.00,7.846472e-04
4,3.00,3.349595e-02
5,4.00,1.992657e-01


,alpha,gamma_star_upper_bound_from_truncated_tilt,tilted_last_bin_mass
0,0.05,3.170367,1.583758e-15
1,0.01,2.586875,3.513794e-18


## 5. Summary

In [7]:
summary = dict(spec_sha256=SPEC_SHA, run_time=str(pd.Timestamp.now(tz='UTC').floor('s')), status=SPEC['status'],
               verification=dict(small_designs=int(len(VR)), max_error=float(VR.err.max()), lambda1_relative_difference=float(rel), lambda1_whole_tail_max_relative_difference=float(rel_all), monotone_in_lambda=MONO_OK),
               m0=dict(alerts=int(P0['n']), T_obs=int(T0), exact_p_nb34=float(REF34.exact_p), exact_mean_nb34=float(REF34.exact_mean)),
               sequential=dict(grid=WC[['lam', 'p_worst']].to_dict('records'), lambda_star=STAR.to_dict('records')),
               rosenbaum_tilt=dict(grid=TR.to_dict('records'), gamma_star_upper_bound=GSTAR.to_dict('records'), last_bin_mass=float(LAW60[-1])))
(OUT / 'SEQ_SENS_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o)))
print(json.dumps(summary['sequential'], indent=1, default=str)); print(json.dumps(summary['rosenbaum_tilt']['gamma_star_upper_bound'], indent=1))
print('Done. Outputs in', OUT)

{
 "grid": [
  {
   "lam": 1.25,
   "p_worst": 3.7381572739392405e-05
  },
  {
   "lam": 1.5,
   "p_worst": 0.0022425532725233
  },
  {
   "lam": 1.6118548977,
   "p_worst": 0.0084929107410147
  },
  {
   "lam": 1.6118548977,
   "p_worst": 0.0084929107410147
  },
  {
   "lam": 1.6410983705,
   "p_worst": 0.0115366246412772
  },
  {
   "lam": 1.6410983705,
   "p_worst": 0.0115366246412772
  },
  {
   "lam": 1.6708724001,
   "p_worst": 0.0155025110942443
  },
  {
   "lam": 1.6708724001,
   "p_worst": 0.0155025110942443
  },
  {
   "lam": 1.7320508076,
   "p_worst": 0.0270905254349103
  },
  {
   "lam": 1.795469241,
   "p_worst": 0.0453008794999131
  },
  {
   "lam": 1.828043982,
   "p_worst": 0.0576154718659261
  },
  {
   "lam": 1.8612097182,
   "p_worst": 0.0724688528828464
  },
  {
   "lam": 2.0,
   "p_worst": 0.1624052306172861
  },
  {
   "lam": 3.0,
   "p_worst": 0.959929088616225
  }
 ],
 "lambda_star": [
  {
   "alpha": 0.05,
   "lambda_star": 1.7954692410260404,
   "bracket_lo":